In [1]:
from pathlib import Path

from caf.base import DVector, ZoningSystem
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

plt.style.use(r'https://raw.githubusercontent.com/Transport-for-the-North/caf.viz/main/src/caf/viz/tfn.mplstyle')

In [2]:
base_year_folder = Path(r"F:\Working\Land-Use\NorCOM 2023 Application (GOR adjustments)")
forecast_folder = Path(r"F:\Working\Land-Use\forecast_population_norcom_test\02_Final Outputs")
output_folder = forecast_folder / "Forecast Analysis"
output_folder.mkdir(exist_ok=True)

In [3]:
gors = ["EM", "EoE", "Lon", "NE", "NW", "Scotland", "SE", "SW", "Wales", "WM", "YH"]

In [4]:
for gor in gors:
    gor_folder = output_folder / gor
    gor_folder.mkdir(exist_ok=True)
    
    results = []
    # get base year data and summarise
    base_year = DVector.load(base_year_folder / f"Output P13.3_{gor}.hdf")
    base = base_year.aggregate(["car_availability"]).data.T
    base.to_csv(gor_folder / f"{gor}_base.csv")
    results.append(
        pd.DataFrame({"year": [2023], "no car hh": [base[1].sum()], "1 car hh": [base[2].sum()], "2+ car hh": [base[3].sum()], "total": [base.sum().sum()]})
    )

    # get forecast data and summarise
    for forecast_file in forecast_folder.glob(f"Output Households_{gor}_*.hdf"):
        year = int(forecast_file.with_suffix("").name.split("_")[-1])
        forecast_year = DVector.load(forecast_file)
        forecast = forecast_year.aggregate(["car_availability"]).data.T
        forecast.to_csv(gor_folder / f"{gor}_forecast_{year}.csv")
        results.append(
            pd.DataFrame({"year": [year], "no car hh": [forecast[1].sum()], "1 car hh": [forecast[2].sum()], "2+ car hh": [forecast[3].sum()], "total": [forecast.sum().sum()]})
        )

    # combine all year results to single dataframe
    result = pd.concat(results)

    # calculate proportions
    result["no car hh proportion"] = result["no car hh"] / result["total"]
    result["1 car hh proportion"] = result["1 car hh"] / result["total"]
    result["2+ car hh proportion"] = result["2+ car hh"] / result["total"]

    # create plot of total households and proportions by different car available levels
    fig, axes = plt.subplots(nrows=2, sharex=True, sharey=False, figsize=(8, 10))
    result.plot.line(x="year", y="total", ax=axes[0], color="C3")
    result.plot.line(x="year", y="no car hh", ax=axes[0], color="C0")
    result.plot.line(x="year", y="1 car hh", ax=axes[0], color="C1")
    result.plot.line(x="year", y="2+ car hh", ax=axes[0], color="C2")
    result[["year", "no car hh proportion", "1 car hh proportion", "2+ car hh proportion"]].plot.area(x="year", ax=axes[1])
    
    axes[0].set_title(f"{gor.upper()} Car Ownership Results")
    axes[0].set_ylabel("Total Households")
    axes[1].set_ylabel("Proportion of Households")
    
    axes[0].yaxis.set_major_formatter(
        matplotlib.ticker.FuncFormatter(lambda x, p: format(int(x), ','))
    )
    axes[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
    axes[1].set_xlabel("Year")
    
    axes[1].set_xlim([result["year"].min() - 3, result["year"].max() + 3])
    axes[1].set_ylim([0, 1])

    # save figure
    fig.savefig(output_folder / f"{gor}_forecast_results.png")
    plt.close()